In [21]:
import warnings
warnings.filterwarnings('ignore')

import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

from xgboost import XGBClassifier

from imblearn.over_sampling import SMOTE

from sklearn.metrics import (
    accuracy_score, 
    confusion_matrix, 
    classification_report, 
    roc_auc_score,
    f1_score,
    recall_score
)

In [22]:
df = pd.read_csv("../Data/Preprocessed_data.csv")
df.head()

,person_age,person_income,person_monthly_income,person_emp_length,emp_stability,loan_grade,loan_amnt,interest_loan_ratio,total_debt_load,loan_int_rate,...,debt_burden_index,cb_person_cred_hist_length,person_home_ownership_OTHER,person_home_ownership_OWN,person_home_ownership_RENT,loan_intent_EDUCATION,loan_intent_HOMEIMPROVEMENT,loan_intent_MEDICAL,loan_intent_PERSONAL,loan_intent_VENTURE
0,3.091042,9.169623,6.685861,1.791759,0.215111,1,6.908755,4.722064,0.113329,2.496506,...,0.113329,1.098612,0,1,0,1,0,0,0,0
1,3.258097,9.169623,6.685861,0.693147,0.039221,2,8.612685,6.563644,0.500775,2.629728,...,0.500775,1.386294,0,0,0,0,0,1,0,0
2,3.178054,11.089821,8.605081,1.609438,0.157004,2,10.463132,8.581388,0.482426,2.786861,...,0.482426,1.098612,0,0,1,0,0,1,0,0
3,3.218876,10.904138,8.419433,2.197225,0.285179,2,10.463132,8.516293,0.553885,2.725890,...,0.553885,1.609438,0,0,1,0,0,1,0,0
4,3.091042,9.200391,6.716595,1.098612,0.095310,0,7.824446,5.190175,0.239017,2.096790,...,0.239017,1.098612,0,1,0,0,0,0,0,1


In [23]:
x = df.drop(['loan_status'], axis=1)
y = df['loan_status']

In [24]:
xtrain, xtest, ytrain, ytest = train_test_split(x, y, test_size=0.2, random_state=42)

In [25]:
algorithms = {
    "LogisticRegression": LogisticRegression(),
    "RandomForestClassifier": RandomForestClassifier(),
    "XGBClassifier": XGBClassifier()
}

# model training without scalling

In [26]:
result_list = []

for model_name, models in algorithms.items():
    model = models.fit(xtrain, ytrain)
    y_pred = model.predict(xtest)

    result_list.append({
        'Model': model_name,
        'Train Score': model.score(xtrain, ytrain),
        'Test Score': model.score(xtest, ytest),
        'accuracy_score': accuracy_score(ytest, y_pred),
        'Confusion Matrix': confusion_matrix(ytest, y_pred)
    })

result = pd.DataFrame(result_list)

In [27]:
print("Without Scalling")
result.sort_values(by='accuracy_score', ascending=False)

Without Scalling


,Model,Train Score,Test Score,accuracy_score,Confusion Matrix
2,XGBClassifier,0.960325,0.939226,0.939226,"[[4918, 35], [348, 1001]]"
1,RandomForestClassifier,1.000000,0.933831,0.933831,"[[4918, 35], [382, 967]]"
0,LogisticRegression,0.853600,0.867344,0.867344,"[[4749, 204], [632, 717]]"


In [28]:
scale_xtrain = xtrain.copy()
scale_xtest = xtest.copy()

In [29]:
scaler = StandardScaler()
scale_xtrain = scaler.fit_transform(scale_xtrain)
scale_xtest = scaler.transform(scale_xtest)

In [30]:
smote = SMOTE()
scale_xtrain_res, scale_ytrain_res = smote.fit_resample(scale_xtrain, ytrain)

In [34]:
result_list_2 = []

for model_name, models in algorithms.items():
    model = models.fit(scale_xtrain_res, scale_ytrain_res)
    y_pred = model.predict(scale_xtest)

    result_list_2.append({
        'Model': model_name,
        'Train Score': model.score(scale_xtrain_res, scale_ytrain_res),
        'Test Score': model.score(scale_xtest, ytest),
        'Accuracy': accuracy_score(ytest, y_pred),
        'Confusion Matrix': confusion_matrix(ytest, y_pred)
    })

result_2 = pd.DataFrame(result_list_2)

In [36]:
print("With Scalling")
result_2.sort_values(by='Accuracy', ascending=False)

With Scalling


,Model,Train Score,Test Score,Accuracy,Confusion Matrix
2,XGBClassifier,0.971541,0.937956,0.937956,"[[4916, 37], [354, 995]]"
1,RandomForestClassifier,1.000000,0.933989,0.933989,"[[4879, 74], [342, 1007]]"
0,LogisticRegression,0.798486,0.812758,0.812758,"[[4042, 911], [269, 1080]]"
